# Text Moderation — TF toxicity classifier (Reddit IRL + lexicons)

Labels a comment toxic/clean for `app/moderation_engine.py`
`/api/v1/moderation/text`. Ground truth is bootstrapped from three local corpora:
the Reddit r/IRL comment stream (`the-reddit-irl-dataset-comments.csv`), the
profanity lexicon (`profanity_en.csv`), and Gen-Z slang intensity
(`genz_slang_usage_2020_2025.csv`). Jigsaw (HF `oxford-ds/toxicity`) is the gold
external source to swap in when an internet connection is available. The trained
model is exported as an ONNX artifact (token-ID input) plus the persisted
text-vectorizer vocabulary, so the serving layer can reproduce tokenisation.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'tensorflow_text'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

SCALE = os.environ.get('BUDDY_SCALE', 'demo')   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


In [ ]:
# Data + bootstrapped labels (lexicon/harsh-slang prior)
import os
import numpy as np
import pandas as pd
from buddy_data import profanity, genz_slang, reddit_irl_comments

N = {'smoke': 4_000, 'demo': 80_000, 'full': 500_000}[SCALE]
USE_SLANG = os.environ.get('BUDDY_SLANG', '1') == '1'

prof = profanity()
slang = genz_slang()

# 2.3 GB CSV — stream only the body column in chunks (bounded memory)
import pandas as pd
from pathlib import Path
rng = np.random.default_rng(42)
chunks = pd.read_csv(Path('../data/the-reddit-irl-dataset-comments.csv'),
                     usecols=['body'], chunksize=250_000, low_memory=False)
texts, want = [], N
for ci, chunk in enumerate(chunks):
    col = chunk['body'].dropna().astype(str)
    texts.append(col.sample(n=min(want, len(col)), random_state=42 + ci))
    want -= len(texts[-1])
    if want <= 0:
        break
texts = pd.concat(texts).tolist()[:N]
print('sampled IRL bodies:', len(texts))

bad = set()
for col in ('canonical_form_1', 'canonical_form_2', 'canonical_form_3'):
    bad |= {w.lower() for w in prof[col].dropna().astype(str)}
hard_slang = set(
    slang.loc[(slang['sentiment_score'] <= -0.2) | (slang['intensity_score'] >= 0.8),
              'slang_term'].dropna().str.lower()
)

def to_label(s: str) -> float:
    s = s.lower()
    if any(w in s for w in bad):
        return 1.0
    if USE_SLANG and any(w in s for w in hard_slang):
        return 0.8
    return 0.0

df = pd.DataFrame({'text': texts})
df['label'] = df['text'].map(to_label)
print('n =', len(df), '| label dist:', df['label'].value_counts().round(3).to_dict())
print('| toxic rate:', round((df['label'] > 0).mean(), 4))

In [ ]:
# Hold-out split (natural skew for honest AUC) + balanced train set
from sklearn.model_selection import train_test_split
train, val = train_test_split(df, test_size=0.2, random_state=42, stratify=(df['label'] > 0))

pos = train[train['label'] > 0]
neg_pool = train[train['label'] == 0]
neg = neg_pool.sample(n=min(len(pos) * 4, len(neg_pool)), random_state=42)
train = pd.concat([pos, neg]).sample(frac=1, random_state=42)
print('train:', train.shape, 'toxic frac:', round((train['label'] > 0).mean(), 3),
      '| val:', val.shape, 'toxic frac:', round((val['label'] > 0).mean(), 4))

In [ ]:
# Vectorizer + BiLSTM classifier (TF/Keras)
import tensorflow as tf

MAX_TOKENS, SEQ, EMB, HID = 40_000, 128, 96, 48
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_sequence_length=SEQ, standardize='lower_and_strip_punctuation')
vectorizer.adapt(np.array(train['text']))
print('vocab size:', vectorizer.vocabulary_size())

inp = tf.keras.Input(shape=(SEQ,), dtype='int64')
x = tf.keras.layers.Embedding(MAX_TOKENS + 2, EMB)(inp)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(HID, return_sequences=True))(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
m = tf.keras.Model(inp, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
m.summary()

In [ ]:
# Train (tokenize once, batch on CPU)
import time
Xtr = vectorizer(np.array(train['text'])).numpy()
Ytr = train['label'].to_numpy()
Xv = vectorizer(np.array(val['text'])).numpy()
Yv = val['label'].to_numpy()

EPOCHS = {'smoke': 1, 'demo': 3, 'full': 6}[SCALE]
t0 = time.time()
hist = m.fit(Xtr, Ytr, epochs=EPOCHS, batch_size=256,
             validation_data=(Xv, Yv), verbose=1)
print(f'train {time.time()-t0:.0f}s')

In [ ]:
# Evaluate: AUC / average precision + threshold calibration (F1)
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

Yb = (Yv > 0).astype(int)                  # metrics need binary labels (0.8 = soft)
p = m.predict(Xv, batch_size=512)[:, 0]
if Yb.sum() == 0 or len(np.unique(Yb)) < 2:
    print('WARNING: val split has no positives (natural skew). Metrics undefined — '
          'raise N / BUDDY_SCALE for real signal.')
    auc = ap = float('nan'); best, f1_50 = 0.5, float('nan')
else:
    auc = roc_auc_score(Yb, p)
    ap = average_precision_score(Yb, p)
    prec, rec, thr = precision_recall_curve(Yb, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    best = float(thr[np.argmax(f1[:-1])])
    f1_50 = f1_score(Yb, (p > 0.5).astype(int))
    print(f'val AUC={auc:.3f} AP={ap:.3f}')
    print(f'@0.5      P={precision_score(Yb,(p>0.5).astype(int)):.3f} '
          f'R={recall_score(Yb,(p>0.5).astype(int)):.3f} F1={f1_50:.3f}')
    print(f'@F1-best  threshold={best:.3f} '
          f'F1={f1_score(Yb,(p>best).astype(int)):.3f}')

In [ ]:
# Check agreement against a held-out lexicon test (sanity)
import json
probe = pd.DataFrame({
    'text': ['i love this recipe, thanks!', 'shut up you stupid idiot', 'great post!',
             'this is the worst thing ive ever seen', 'hahaha lol'],
})
Xp = vectorizer(np.array(probe['text'])).numpy()
probe['score'] = m.predict(Xp)[:, 0].round(3)
print(probe.to_string(index=False))

### Export contract (consumed by the AI service)

The cells below write `../models/toxicity_classifier.onnx` and its dynamic-INT8 quantized copy
`toxicity_classifier_int8.onnx`. `app/ml/serving.py::load_preferred('toxicity_classifier')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [ ]:
# Export ONNX (+ INT8) + persist the vectorizer vocab for serving
from pathlib import Path
import json
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(
    m, Path('../models'), 'toxicity_classifier', '1.0.0',
    input_signature=[tf.TensorSpec((None, SEQ), tf.int64, name='input_ids')])
q = quantize_dynamic_onnx(onnx)

vocab = {'max_tokens': MAX_TOKENS, 'sequence_length': SEQ,
         'vocabulary': vectorizer.get_vocabulary(), 'threshold': best}
(Path('../models') / 'toxicity_vectorizer.json').write_text(json.dumps(vocab))

mlflow_log({'name': 'toxicity_classifier', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'val_auc': float(auc), 'val_ap': float(ap), 'threshold': float(best)}})
print('exported', q)